# A/B Test the Model, Keep the Prompt

Evaluate one conversation, then 50 conversation threads asynchronously, changing only the model through OpenRouter.

This follows [OpenRouter: Setup, Costs and a Simple Agent](openrouter_features_and_agent.ipynb).

**Same prompt. Same conversation. Same tools. Different model, potentially different agent trajectory.** A gateway adds model selection as another optimisation lever alongside prompt engineering.

## 1. OpenRouter and AgentEvals

[AgentEvals](https://github.com/langchain-ai/agentevals) is an open-source evaluation library from **LangChain**. It evaluates an agent's **trajectory**, the intermediate steps and tool calls, against a reference. It supports deterministic matching and model-based judges. Here we use deterministic matching with the native OpenRouter Python and agent SDKs, so no judge model or extra key is needed.

Run in Jupyter or Colab with Python 3.10+. Use your [OpenRouter key](https://openrouter.ai/settings/keys) and credits. The single example makes two paid agent runs. The optional batch makes **100 agent runs: the same 50 threads per model**, with several model requests per run.

In [ ]:
"""Compare OpenRouter models using fixed prompts, tools and ground truth."""
# Jupyter supports top-level await; Pylint replaces pip magics with pass.
# pylint: disable=await-outside-async,unnecessary-pass
%pip install openrouter==1.1.22 openrouter-agent-sdk==0.8.0 agentevals==0.0.9 pandas statsmodels==0.15.0 --upgrade --quiet

In [ ]:
import asyncio
import json
import os
import re
from getpass import getpass
from math import ceil
from statistics import NormalDist
from time import perf_counter

import pandas as pd
from IPython.display import display
from pydantic import BaseModel
from openrouter import OpenRouter
from openrouter_agent import call_model, tool, step_count_is
from agentevals.trajectory.match import create_trajectory_match_evaluator
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.contingency_tables import mcnemar

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter API key: ")
router = OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"])

### Models: edit only this cell

Latest standard GPT Sol and Gemini Flash releases verified on 8 October 2026: [GPT-6.1 Sol](https://openrouter.ai/openai/gpt-6.1-sol) and [Gemini 3.8 Flash](https://openrouter.ai/google/gemini-3.8-flash). Use explicit model IDs to keep the experiment identifiable.

GPT-6.1 Sol does not expose temperature control. We use shared supported settings and low reasoning effort for both models.

In [ ]:
MODELS = [
    "openai/gpt-6.1-sol",
    "google/gemini-3.8-flash",
]

## 2. Ground truth and three evals

**Ground truth** is the expected behaviour and facts we define before seeing outputs. An **eval** compares the actual run with those expectations.

| Eval | Pass rule |
| --- | --- |
| Tool count | At least `min_tool_calls` completed lookups, here 3 |
| Expected tools | Correct conversation and order lookups, plus at least one of `get_policy` or `search_policy`, with the correct arguments |
| Summary facts | Contains the expected customer, order, issue, status and return window |

Tool calls belong to the agent's run. The final summary contains the retrieved facts. For model selection, our primary metric is **tool success**: the first two evals both pass on a thread. Summary coverage stays separate.

In [ ]:
NAMES = ["Mina Patel", "Owen Clarke", "Zara Khan", "Leo Evans", "Nia Thomas",
         "Amir Ali", "Ruby Jones", "Theo Scott", "Eva Green", "Noah Brown"]
SCENARIOS = [
    ("damaged", "delivered", "30 days"),
    ("wrong size", "delivered", "14 days"),
    ("wrong colour", "delivered", "30 days"),
    ("wrong item", "returned", "60 days"),
    ("missing part", "delivered", "14 days"),
]


def make_case(index):
    """Create a fictional thread and its human-defined ground truth."""
    customer = NAMES[index // 5]
    issue, status, window = SCENARIOS[index % 5]
    conversation_id, order_id = f"C{104 + index}", f"O{104 + index}"
    conversation = (
        f"Customer: I am {customer}. Order {order_id} has this issue: {issue}.\n"
        "Agent: I will check the order and returns policy."
    )
    return {
        "id": conversation_id,
        "records": {
            "get_conversation": {
                conversation_id: {
                    "transcript": conversation, "order_id": order_id, "policy_id": "returns",
                },
            },
            "get_order": {order_id: {"status": status}},
            "get_policy": {"returns": {"return_window": window}},
            "search_policy": {"returns": {"return_window": window}},
        },
        "truth": {
            "min_tool_calls": 3,
            "required_calls": [("get_conversation", conversation_id), ("get_order", order_id)],
            "one_of_calls": [("get_policy", "returns"), ("search_policy", "returns")],
            "summary_facts": [
                [customer], [order_id], [issue], [status],
                [window, window.replace(" days", "-day")],
            ],
        },
    }

EXAMPLE = make_case(0)
display(EXAMPLE)

## 3. Record real tool executions

Each run gets its own tools and call log, so async runs cannot mix their traces. Record a call after a successful lookup. A requested tool or its name in the summary does not count as an executed call.

In [ ]:
class LookupInput(BaseModel):
    """Identifier of the record to retrieve."""
    id: str

DESCRIPTIONS = {
    "get_conversation": "Retrieve a conversation by its conversation ID.",
    "get_order": "Retrieve the order status by the order ID found in the conversation.",
    "get_policy": "Retrieve a policy by the policy ID found in the conversation.",
    "search_policy": "Find a policy by the policy ID found in the conversation.",
}


def make_lookup_tool(name, records, completed_calls):
    """Create a read-only local tool with a log scoped to this run."""
    def lookup(params, _context):
        record = records[params.id]
        completed_calls.append({"name": name, "arguments": {"id": params.id}})
        return record
    return tool(name=name, description=DESCRIPTIONS[name],
                input_schema=LookupInput, execute=lookup)

## 4. Evaluate the trajectory and summary

Our `has_tool_call` helper wraps AgentEvals' built-in `superset` matcher. It checks that a reference call appears anywhere in the run, with **exact arguments**, while allowing extra calls and different ordering.

The summary check accepts listed phrases regardless of case. It measures coverage and can miss contradictions, so it is a small teaching eval rather than a full factuality assessment.

In [ ]:
match_tools = create_trajectory_match_evaluator(
    trajectory_match_mode="superset", tool_args_match_mode="exact",
)


def as_trajectory(calls):
    """Convert completed calls to the message format AgentEvals accepts."""
    return [{"role": "assistant", "content": "", "tool_calls": [
        {"id": str(i), "type": "function", "function": {
            "name": call["name"], "arguments": json.dumps(call["arguments"]),
        }} for i, call in enumerate(calls)
    ]}]


def has_tool_call(calls, name, record_id):
    """Match the expected tool name and its arguments."""
    reference = [{"name": name, "arguments": {"id": record_id}}]
    evaluation = match_tools(
        outputs=as_trajectory(calls), reference_outputs=as_trajectory(reference),
    )
    if not isinstance(evaluation, dict):
        raise TypeError("Expected a single trajectory match result.")
    return evaluation["score"]


def evaluate_run(summary, calls, truth):
    """Apply the three ground-truth checks."""
    return {
        "tool_count": len(calls) >= truth["min_tool_calls"],
        "expected_tools": all(has_tool_call(calls, *args) for args in truth["required_calls"])
            and any(has_tool_call(calls, *args) for args in truth["one_of_calls"]),
        "summary_facts": all(
            any(re.search(r"\b" + re.escape(term) + r"\b", summary, re.I)
                for term in alternatives)
            for alternatives in truth["summary_facts"]
        ),
    }

## 5. Start with one conversation thread

The prompt and tool schemas stay fixed. The model chooses its trajectory. The step limit caps model turns and does not force a minimum number of tool calls.

In [ ]:
PROMPT = """Summarise the support conversation using only retrieved facts.
Before answering, retrieve the conversation, its order record and applicable policy.
Include customer name, order ID, issue, order status and return window.
Keep the summary short and preserve those facts as returned."""
SETTINGS = {"instructions": PROMPT, "max_output_tokens": 2048,
            "reasoning": {"effort": "low"}, "stop_when": step_count_is(5)}


async def run_case(model, case):
    """Run one model on one thread and retain its own evidence."""
    completed_calls = []
    tools = [make_lookup_tool(name, records, completed_calls)
             for name, records in case["records"].items()]
    start = perf_counter()
    summary, error = "", ""
    try:
        result = call_model(router, {"model": model, **SETTINGS,
            "input": f"Summarise support conversation {case['id']}.", "tools": tools})
        summary = await asyncio.wait_for(result.get_text(), timeout=120)
        checks = evaluate_run(summary, completed_calls, case["truth"])
    # Keep each failed run in the results, even for unexpected provider errors.
    except Exception as exc:  # pylint: disable=broad-exception-caught
        error = type(exc).__name__
        checks = dict.fromkeys(["tool_count", "expected_tools", "summary_facts"], False)
    return {"model": model, "thread_id": case["id"], **checks,
            "tool_success": checks["tool_count"] and checks["expected_tools"],
            "tool_calls": len(completed_calls), "calls": completed_calls,
            "summary": summary, "error": error, "seconds": perf_counter() - start}

In [ ]:
single_runs = [await run_case(model, EXAMPLE) for model in MODELS]
for run in single_runs:
    print(
        f"\n{run['model']}\nTools: {[call['name'] for call in run['calls']]}"
        f"\nSummary: {run['summary']}"
    )
COLUMNS = [
    "model", "thread_id", "tool_count", "expected_tools",
    "summary_facts", "tool_success", "tool_calls", "error",
]
display(pd.DataFrame(single_runs)[COLUMNS])

## 6. Evaluate 50 threads asynchronously

One example helps explain a run. Fifty different threads give us more observations. Both models get the **same 50 inputs** and the same prompt. `asyncio.gather` runs them concurrently; a semaphore allows at most six agent runs at once.

Set `RUN_BATCH = True` to make 100 paid agent runs. Failed runs remain in the results and count as failures in the operational tool-success rate; inspect the error column before attributing differences to the models.

These fictional examples vary the customer, issue, order status and policy. They demonstrate the workflow. For population claims, replace them with independently sampled, representative, held-out conversation threads with reviewed ground truth.

In [ ]:
RUN_BATCH = False
CONCURRENCY = 6
CASES = [make_case(index) for index in range(50)]


async def run_batch(cases):
    """Evaluate the same cases for every model with bounded concurrency."""
    semaphore = asyncio.Semaphore(CONCURRENCY)
    async def limited(model, case):
        async with semaphore:
            return await run_case(model, case)
    return await asyncio.gather(*(limited(model, case) for case in cases for model in MODELS))

batch_results = None
if RUN_BATCH:
    batch_runs = await run_batch(CASES)
    batch_results = pd.DataFrame(batch_runs)
    display(batch_results[COLUMNS])
else:
    print("50 cases ready. Set RUN_BATCH = True, then rerun this cell.")

## 7. How certain can we be?

Count **one pass/fail observation per thread per model**. Three tool calls in each of 50 threads still give us **50 observations**, because calls within a thread depend on one another.

Use a **95% Wilson confidence interval** for each model's tool-success rate. Under independent, representative sampling, this procedure has approximately 95% coverage over repeated samples. It does not assign a 95% probability that a model is better.

For example, **45/50 successes = 90%, with a 95% interval of about 79% to 96%**. Even **50/50** gives an interval of about **93% to 100%**, so it cannot establish perfect reliability. [Wilson intervals: NIST](https://www.itl.nist.gov/div898/handbook/prc/section2/prc241.htm)

In [ ]:
def tool_success_report(results):
    """Use threads, rather than individual tool calls, as the sample size."""
    rows = []
    for model, group in results.groupby("model", sort=False):
        successes, n = int(group["tool_success"].sum()), len(group)
        low, high = proportion_confint(successes, n, alpha=0.05, method="wilson")
        rows.append({"model": model, "threads": n, "successes": successes,
            "tool_success_rate": f"{successes / n:.1%}", "95% CI": f"{low:.1%} to {high:.1%}",
            "errors": int(group["error"].ne("").sum())})
    return pd.DataFrame(rows).set_index("model")

if batch_results is not None:
    display(tool_success_report(batch_results))

# Illustrations only, not scores returned by the models.
for example_successes in [45, 50]:
    example_low, example_high = proportion_confint(example_successes, 50, method="wilson")
    print(
        f"Illustration: {example_successes}/50 -> {example_successes / 50:.0%}, "
        f"95% CI {example_low:.1%} to {example_high:.1%}"
    )

### What sample size is adequate?

Choose the precision you need. For estimating one model's success rate at 95% confidence, a conservative planning rule is `n ≈ 1.96² × 0.5 × 0.5 / margin²`. The margin is in percentage points. About **97 threads** targets ±10 points, **385** targets ±5, and **1,068** targets ±3. This is an approximation for independent, representative samples, using the most variable success rate, 50%.

Fifty threads are useful for spotting large failures, but generally too few to settle small differences. Estimating reliability and detecting a difference between models are separate questions. Comparison sample size also depends on the smallest difference you care about, how often the models disagree, and the desired statistical power.

Because both models see the same threads, compare **paired outcomes**. The exact McNemar test below uses only threads where one model passed and the other failed. A small p-value indicates evidence of a difference under its assumptions, rather than a probability that one model is best. Confirm a selected model on fresh held-out data. [Paired test](https://www.statsmodels.org/stable/generated/statsmodels.stats.contingency_tables.mcnemar.html)

In [ ]:
z = NormalDist().inv_cdf(0.975)
display(pd.DataFrame([{"target_margin": f"±{margin * 100:.0f} percentage points",
    "approx_threads_per_model": ceil(z**2 * 0.25 / margin**2)}
    for margin in [0.10, 0.05, 0.03]]))

if batch_results is not None and len(MODELS) == 2:
    paired = batch_results.pivot(index="thread_id", columns="model", values="tool_success")
    a, b = MODELS
    a_only = int((paired[a] & ~paired[b]).sum())
    b_only = int((~paired[a] & paired[b]).sum())
    both = int((paired[a] & paired[b]).sum())
    neither = int((~paired[a] & ~paired[b]).sum())
    # Statsmodels returns a result object with dynamically added attributes.
    paired_test = mcnemar([[both, a_only], [b_only, neither]], exact=True)
    p_value = getattr(paired_test, "pvalue")
    print(f"{a} only: {a_only}; {b} only: {b_only}; exact paired p-value: {p_value:.4f}")

## Summary

- OpenRouter makes model selection another optimisation lever: **fix the prompt, conversations, tools and evals, then change the model**.
- AgentEvals checks the recorded tool trajectory. Ground truth also lets us check summary coverage.
- Inspect one thread, then evaluate the same 50 threads per model asynchronously. Each run keeps its own evidence.
- Report tool-success rates with confidence intervals. Representative sampling and paired comparisons matter as much as the sample count.